In [44]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
import matplotlib.pyplot as plt


In [7]:
distancia = pd.read_excel('exerc1.xlsx',sheet_name='distancia')
producao = pd.read_excel('exerc1.xlsx',sheet_name='producao')

In [8]:
distancia = distancia.set_index('fabricas')
producao = producao.set_index('fabricas')

In [30]:
dict_distancia = distancia.T.to_dict()
dict_distancia['f1']

{'c1': 800, 'c2': 1300, 'c3': 400, 'c4': 700}

In [18]:
dict_producao = producao['quantidade_produzida'].to_dict()
dict_producao

{'f1': 12, 'f2': 17, 'f3': 11}

In [81]:
model = pyo.ConcreteModel()

model.fabricas = pyo.Set(initialize=dict_producao.keys())
model.cds= pyo.Set(initialize=distancia.columns)
model.x = pyo.Var(model.fabricas, model.cds, bounds=(0,None),domain=NonNegativeReals)

model.custo_transporte = pyo.Var(initialize=0.50)

def objetivo(model):
    return sum(
        (model.x[fabrica, cd] * dict_distancia[fabrica][cd])/2 for fabrica in model.fabricas for cd in model.cds
        )
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.minimize)

def restricao_minima_cd(model, cd):

    return sum(
        model.x[fabrica, cd] for fabrica in model.fabricas
    )  >= 10
model.restricao_minima_cd = pyo.Constraint(model.cds, rule=restricao_minima_cd)

def restricao_maxima_fabrica(model, fabrica):
    return sum(
        model.x[fabrica, cd] for cd in model.cds
    )  == dict_producao[fabrica]
model.restricao_maxima_fabrica = pyo.Constraint(model.fabricas, rule=restricao_maxima_fabrica)

In [82]:
model.pprint()

2 Set Declarations
    cds : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'c1', 'c2', 'c3', 'c4'}
    fabricas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'f1', 'f2', 'f3'}

2 Var Declarations
    custo_transporte : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :  None :   0.5 :  None : False : False :  Reals
    x : Size=12, Index=fabricas*cds
        Key          : Lower : Value : Upper : Fixed : Stale : Domain
        ('f1', 'c1') :     0 :  None :  None : False :  True : NonNegativeReals
        ('f1', 'c2') :     0 :  None :  None : False :  True : NonNegativeReals
        ('f1', 'c3') :     0 :  None :  None : False :  True : NonNegativeReals
        ('f1', 'c4') :     0 :  None :  None : False :  True : NonNegativeReals
        ('f2', 'c1') :     0 :  None :  None : False

In [83]:
opt = SolverFactory('cplex')
# opt.options['optimality target'] = 3
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpe3fjup7f.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpt_uw7c_a.pyomo.lp' read.
Read time = 0.02 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpt_uw7c_a.pyomo.lp
Objective sense      : Minimize
Variables            :      12
Objective nonzeros   :      12
Linear constraints   :       7  [Greater: 4,  Equal: 3]
  Nonzeros           :      24
  RHS nonzeros       :       7

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 200.0000         Max   

In [84]:
model.x.pprint()

x : Size=12, Index=fabricas*cds
    Key          : Lower : Value : Upper : Fixed : Stale : Domain
    ('f1', 'c1') :     0 :   0.0 :  None : False : False : NonNegativeReals
    ('f1', 'c2') :     0 :   0.0 :  None : False : False : NonNegativeReals
    ('f1', 'c3') :     0 :   2.0 :  None : False : False : NonNegativeReals
    ('f1', 'c4') :     0 :  10.0 :  None : False : False : NonNegativeReals
    ('f2', 'c1') :     0 :   0.0 :  None : False : False : NonNegativeReals
    ('f2', 'c2') :     0 :   9.0 :  None : False : False : NonNegativeReals
    ('f2', 'c3') :     0 :   8.0 :  None : False : False : NonNegativeReals
    ('f2', 'c4') :     0 :   0.0 :  None : False : False : NonNegativeReals
    ('f3', 'c1') :     0 :  10.0 :  None : False : False : NonNegativeReals
    ('f3', 'c2') :     0 :   1.0 :  None : False : False : NonNegativeReals
    ('f3', 'c3') :     0 :   0.0 :  None : False : False : NonNegativeReals
    ('f3', 'c4') :     0 :   0.0 :  None : False : False : NonNega

In [87]:
pyo.value(model.objetivo)

16200.0